In [ ]:
!pip install paho-mqtt

In [ ]:
import time
import random
import json
import paho.mqtt.client as mqtt

# Configurações do Broker MQTT (Público)
BROKER = "broker.hivemq.com"
PORT = 1883
TOPIC_BASE = "inatel/smartzone/setorA"

def on_connect(client, userdata, flags, rc, properties=None):
    if rc == 0:
        print("[MQTT] Conectado com sucesso ao Broker HiveMQ!")
    else:
        print(f"[MQTT] Falha na conexão. Código: {rc}")

client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, "SmartZone_Edge_Node")
client.on_connect = on_connect
client.connect(BROKER, PORT, 60)
client.loop_start()

# Configuração de 8 vagas do estacionamento
vagas = [f"VAGA_{str(i).zfill(2)}" for i in range(1, 9)]
estado_vagas = {
    vaga: {
        "status": "LIVRE",
        "veiculo_id": "---",
        "tempo_ocupado_min": 0,
        "bateria_sensor_pct": 98
    } for vaga in vagas
}

print("=== SISTEMA SMARTZONE INATEL - SIMULAÇÃO MULTI-VAGAS ATIVA ===")

try:
    while True:
        # Altera dinamicamente o estado de uma vaga aleatória
        vaga_alvo = random.choice(vagas)

        if estado_vagas[vaga_alvo]["status"] == "LIVRE":
            if random.random() > 0.3:  # 70% de chance de ocupar
                estado_vagas[vaga_alvo]["status"] = "OCUPADO"
                estado_vagas[vaga_alvo]["veiculo_id"] = f"ABC-{random.randint(1000, 9999)}"
                estado_vagas[vaga_alvo]["tempo_ocupado_min"] = random.randint(2, 60)
        else:
            if random.random() > 0.5:  # 50% de chance de desocupar
                estado_vagas[vaga_alvo]["status"] = "LIVRE"
                estado_vagas[vaga_alvo]["veiculo_id"] = "---"
                estado_vagas[vaga_alvo]["tempo_ocupado_min"] = 0

        # Computa métricas globais do Digital Twin
        total_vagas = len(vagas)
        ocupadas = sum(1 for v in estado_vagas.values() if v["status"] == "OCUPADO")
        taxa = (ocupadas / total_vagas) * 100

        payload_global = {
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
            "setor": "Centro-A_Inatel",
            "metricas": {
                "total_vagas": total_vagas,
                "vagas_disponiveis": total_vagas - ocupadas,
                "vagas_ocupadas": ocupadas,
                "taxa_ocupacao_pct": round(taxa, 2)
            },
            "detalhes_vagas": estado_vagas
        }

        # Publica no MQTT
        client.publish(f"{TOPIC_BASE}/status_global", json.dumps(payload_global))
        print(f"[{time.strftime('%H:%M:%S')}] Ocupação do Setor: {ocupadas}/{total_vagas} ({taxa:.1f}%) -> Publicado!")

        print("\n" + "="*50)
        print(f"[{time.strftime('%H:%M:%S')}] STATUS DA RUA (Ocupação: {taxa:.1f}%)")
        print("="*50)
        for v_id, info in estado_vagas.items():
            status_txt = "🔴 OCUPADO" if info["status"] == "OCUPADO" else "🟢 LIVRE   "
            veiculo = f"Placa: {info['veiculo_id']}" if info["status"] == "OCUPADO" else "Disponível"
            print(f"  {v_id} -> {status_txt} | {veiculo}")
        print("="*50)

        time.sleep(5)

except KeyboardInterrupt:
    client.loop_stop()
    client.disconnect()
    print("\nSimulação encerrada com segurança.")

Streaming output truncated to the last 5000 lines.
  VAGA_08 -> 🔴 OCUPADO | Placa: ABC-8792
[15:01:29] Ocupação do Setor: 2/8 (25.0%) -> Publicado!

[15:01:29] STATUS DA RUA (Ocupação: 25.0%)
  VAGA_01 -> 🟢 LIVRE    | Disponível
  VAGA_02 -> 🟢 LIVRE    | Disponível
  VAGA_03 -> 🟢 LIVRE    | Disponível
  VAGA_04 -> 🔴 OCUPADO | Placa: ABC-3768
  VAGA_05 -> 🟢 LIVRE    | Disponível
  VAGA_06 -> 🟢 LIVRE    | Disponível
  VAGA_07 -> 🟢 LIVRE    | Disponível
  VAGA_08 -> 🔴 OCUPADO | Placa: ABC-8792
[15:01:34] Ocupação do Setor: 2/8 (25.0%) -> Publicado!

[15:01:34] STATUS DA RUA (Ocupação: 25.0%)
  VAGA_01 -> 🟢 LIVRE    | Disponível
  VAGA_02 -> 🟢 LIVRE    | Disponível
  VAGA_03 -> 🟢 LIVRE    | Disponível
  VAGA_04 -> 🔴 OCUPADO | Placa: ABC-3768
  VAGA_05 -> 🟢 LIVRE    | Disponível
  VAGA_06 -> 🟢 LIVRE    | Disponível
  VAGA_07 -> 🟢 LIVRE    | Disponível
  VAGA_08 -> 🔴 OCUPADO | Placa: ABC-8792
[15:01:39] Ocupação do Setor: 3/8 (37.5%) -> Publicado!

[15:01:39] STATUS DA RUA (Ocupação: 37.5%)
